# Lab 12: Template de Projeto Final

## Lab 12: Template de Projeto Final — testado de ponta a ponta

Este notebook é um **esqueleto funcional**, não um exercício com resposta
fixa. Roda com `tiny-gpt2` como placeholder (pra você confirmar que a
estrutura inteira funciona antes de trocar pelos componentes reais do seu
projeto) — troque `MODEL_NAME`, `load_project_dataset()` e `REWARD_FN`
pelo seu caso de uso real.

In [1]:
!pip install -q transformers torch peft datasets

from dataclasses import dataclass, field
from typing import Optional
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

### 1. Config do projeto (Semana 12.1, passos 1-2)

In [2]:
@dataclass
class ProjectConfig:
    name: str = "meu-projeto-de-customizacao"
    model_name: str = "sshleifer/tiny-gpt2"  # TROQUE pelo modelo real do seu projeto
    strategy: str = "sft"  # "sft" | "lora" | "qlora" | "dpo" — troque pra "lora" pra ver a diferença
    lora_r: int = 4
    lora_alpha: int = 16
    lora_target_modules: list = field(default_factory=lambda: ["c_attn"])
    train_steps: int = 20
    learning_rate: float = 5e-3
    eval_prompts: list = field(default_factory=lambda: ["The purpose of this project is"])

config = ProjectConfig()
print(f"✓ Config carregada: {config.name} | estratégia: {config.strategy} | modelo: {config.model_name}")

✓ Config carregada: meu-projeto-de-customizacao | estratégia: sft | modelo: sshleifer/tiny-gpt2


### 2. Dataset do projeto (Semana 12.1, passo 3) — TROQUE por dados reais

In [3]:
def load_project_dataset(n: int = 20):
    """PLACEHOLDER — troque por Semana 4 (curadoria real): dataset
    limpo, deduplicado, formatado no chat template do seu modelo real."""
    raw = load_dataset("databricks/databricks-dolly-15k", split=f"train[:{n}]")
    return [f"Instruction: {ex['instruction']}\nResponse: {ex['response']}" for ex in raw if ex["instruction"] and ex["response"]]

train_texts = load_project_dataset()
print(f"✓ {len(train_texts)} exemplos de treino")

✓ 20 exemplos de treino


### 3. Baseline — SEMPRE meça antes de customizar (Semana 12.1, passo 4)

In [4]:
tokenizer = AutoTokenizer.from_pretrained(config.model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def evaluate(model, prompts, max_new_tokens=20):
    """Harness de avaliação simples — troque por métricas reais do seu
    domínio (accuracy, taxa de conversão, o que for relevante)."""
    model.eval()
    results = []
    for p in prompts:
        inputs = tokenizer(p, return_tensors="pt")
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        results.append(tokenizer.decode(out[0], skip_special_tokens=True))
    return results

baseline_model = AutoModelForCausalLM.from_pretrained(config.model_name)
baseline_outputs = evaluate(baseline_model, config.eval_prompts)
print("📊 BASELINE (antes de qualquer customização):")
for p, o in zip(config.eval_prompts, baseline_outputs):
    print(f"  '{p}' → {o!r}")

📊 BASELINE (antes de qualquer customização):
  'The purpose of this project is' → 'The purpose of this project is stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs'


### 4. Aplicando a estratégia escolhida (Semana 12.1, passo 5)

In [5]:
def build_model_for_strategy(config: ProjectConfig):
    """Ponto de decisão central do projeto — cada branch aponta pra onde
    você já viu a implementação completa nas semanas anteriores."""
    base = AutoModelForCausalLM.from_pretrained(config.model_name)

    if config.strategy == "sft":
        return base  # full fine-tuning direto — ver Lab 5

    elif config.strategy == "lora":
        lora_config = LoraConfig(
            task_type=TaskType.CAUSAL_LM, r=config.lora_r, lora_alpha=config.lora_alpha,
            lora_dropout=0.05, target_modules=config.lora_target_modules,
        )
        return get_peft_model(base, lora_config)  # ver Lab 6

    elif config.strategy == "qlora":
        raise NotImplementedError("Ver Lab 6, seção 3 — precisa de BitsAndBytesConfig no carregamento do base")

    elif config.strategy == "dpo":
        raise NotImplementedError("Precisa de dataset de preferência (chosen/rejected) — ver Lab 8")

    raise ValueError(f"Estratégia desconhecida: {config.strategy}")

model = build_model_for_strategy(config)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"✓ Modelo preparado com estratégia '{config.strategy}': {trainable:,}/{total:,} parâmetros treináveis ({100*trainable/total:.2f}%)")

✓ Modelo preparado com estratégia 'sft': 102,714/102,714 parâmetros treináveis (100.00%)


### 5. Treino (Semana 12.1, passo 6) — mesmo loop das semanas anteriores

In [6]:
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=config.learning_rate)
model.train()
losses = []

for step in range(config.train_steps):
    text = train_texts[step % len(train_texts)]
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=64)
    outputs = model(**inputs, labels=inputs["input_ids"])
    outputs.loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    losses.append(outputs.loss.item())

print(f"✓ Treino concluído: loss {losses[0]:.3f} → {losses[-1]:.3f}")

✓ Treino concluído: loss 10.828 → 10.758


### 6. Avaliação final — comparação com o baseline (Semana 12.1, passos 8-10)

In [7]:
final_outputs = evaluate(model, config.eval_prompts)

print("=" * 60)
print(f"RELATÓRIO FINAL — {config.name}")
print("=" * 60)
print(f"Estratégia: {config.strategy}")
print(f"Parâmetros treináveis: {trainable:,} ({100*trainable/total:.2f}% do total)")
print(f"Loss: {losses[0]:.3f} → {losses[-1]:.3f}")
print()
for p, before, after in zip(config.eval_prompts, baseline_outputs, final_outputs):
    print(f"Prompt: '{p}'")
    print(f"  ANTES:  {before!r}")
    print(f"  DEPOIS: {after!r}")
print("=" * 60)
print("\n📝 Próximo passo real: troque config.model_name por um modelo de")
print("verdade, load_project_dataset() pelos seus dados curados (Semana 4),")
print("e config.eval_prompts por um conjunto de avaliação representativo")
print("do SEU problema — a estrutura acima já está validada e funcionando.")

RELATÓRIO FINAL — meu-projeto-de-customizacao
Estratégia: sft
Parâmetros treináveis: 102,714 (100.00% do total)
Loss: 10.828 → 10.758

Prompt: 'The purpose of this project is'
  ANTES:  'The purpose of this project is stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs'
  DEPOIS: 'The purpose of this project isructionructionructionructionructionructionructionructionructionructionructionructionructionructionructionructionructionructionructionruction'

📝 Próximo passo real: troque config.model_name por um modelo de
verdade, load_project_dataset() pelos seus dados curados (Semana 4),
e config.eval_prompts por um conjunto de avaliação representativo
do SEU problema — a estrutura acima já está validada e funcionando.


**Resultado esperado:** um relatório completo rodando de ponta a ponta —
config → dataset → baseline → estratégia → treino → avaliação comparativa
— confirmando que o *esqueleto* do pipeline funciona antes de você
investir tempo customizando pra um problema real. Com `strategy="sft"`
(full fine-tuning, o default aqui), a saída ANTES/DEPOIS deve mudar
visivelmente (mesmo comportamento do Lab 5). Se você trocar pra
`strategy="lora"`, é esperado que a saída **não** mude muito — vimos no
Lab 6 que LoRA aplicado só em `c_attn` desse modelo específico tem
gradiente quase nulo; isso não é o template quebrado, é o mesmo efeito já
documentado, e é exatamente o tipo de coisa que checar ANTES/DEPOIS (em
vez de assumir que "rodou = funcionou") existe pra pegar.

---

## 🎓 Fim da Fase 2

Você cobriu, com código real e testado: fundamentos de LLM e Transformers,
como o treino funciona, curadoria de dataset, SFT, LoRA/QLoRA, reasoning,
DPO, GRPO, distillation, MoE, quantização e inferência eficiente — a mesma
cobertura de tópicos do curso pago que motivou esta trilha, sem custo.

**Onde ir a partir daqui:** troque os modelos de brinquedo (`tiny-gpt2`,
`SmolLM2-135M`) por modelos reais numa GPU (Colab gratuito já ajuda muito),
e aplique a um problema que você realmente precisa resolver.